In [2]:
import os, glob, warnings, numpy as np, pandas as pd, psycopg, pyarrow.parquet as pq
from dotenv import load_dotenv
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
load_dotenv("../.env")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, connect_timeout=10)

SEED, CAP, K = 20261030, 200, 40
neg = pd.read_sql("SELECT recommendationid, appid FROM reviews WHERE NOT voted_up AND NOT is_short", conn)
sample = neg.sample(frac=1, random_state=SEED).groupby("appid").head(CAP)     # shuffle, then at most CAP per game
print(f"{len(neg):,} negative non-short reviews; sample {len(sample):,} from {sample.appid.nunique()} games")

156,487 negative non-short reviews; sample 46,435 from 300 games


In [3]:
want = sample.recommendationid.to_numpy()
ids, vecs = [], []
for p in sorted(glob.glob("../data/embeddings/part-*.parquet")):
    t = pq.read_table(p)
    rid = t.column("recommendationid").to_numpy()
    m = np.isin(rid, want)
    if m.any():
        ids.append(rid[m])
        vecs.append(t.column("embedding").combine_chunks().flatten().to_numpy().reshape(-1, 384)[m])
ids, X = np.concatenate(ids), np.vstack(vecs)

meta = (pd.read_sql("""SELECT r.recommendationid, r.appid, g.name, r.review
                       FROM reviews r JOIN games g USING (appid)
                       WHERE r.recommendationid = ANY(%(ids)s)""", conn, params={"ids": ids.tolist()})
          .set_index("recommendationid").loc[ids])
print(X.shape, f"{len(sample) - len(ids)} sampled reviews without a vector (newer than the last embedding run)")

(45633, 384) 802 sampled reviews without a vector (newer than the last embedding run)


In [4]:
km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X)
meta["cluster"] = km.labels_
meta["dist"] = np.linalg.norm(X - km.cluster_centers_[km.labels_], axis=1)    # distance to the cluster's centre

# Words that set each cluster apart: TF-IDF over one combined "document" per cluster
docs = meta.groupby("cluster").review.apply(lambda s: " ".join(s.str.lower()))
tf = TfidfVectorizer(stop_words="english", min_df=2, max_df=0.5, sublinear_tf=True,
                     ngram_range=(1, 2), max_features=30000)
W = tf.fit_transform(docs)
terms = np.array(tf.get_feature_names_out())

rows = []
for c in docs.index:
    m = meta[meta.cluster == c]
    games = m.name.value_counts(normalize=True)
    rows.append({"cluster": c, "reviews": len(m), "games": m.appid.nunique(),
                 "top_game": games.index[0][:30], "top_game_share": round(games.iloc[0], 2),
                 "words": ", ".join(terms[W[c].toarray().ravel().argsort()[::-1][:8]])})
pd.set_option("display.max_colwidth", 150)
summary = pd.DataFrame(rows).sort_values("reviews", ascending=False)
summary

,cluster,reviews,games,top_game,top_game_share,words
20,20,2064,269,MOUSE: P.I. For Hire,0.02,"like graphics, combat locks, hellblade, secret mana, che, ksp, sea stars, yakumo"
4,4,1828,258,Slime Rancher 2,0.03,"olden era, slime rancher, gate enhanced, olden, ddd, rancher, revenant hunter, josee"
26,26,1817,229,Everything is Crab: The Animal,0.03,"vholume, serum, lp, igtap, mace, sts2, heroes spells, cult lamb"
5,5,1733,269,The Witness,0.02,"mode challenge, msq, pressure 10, puzzle games, challenge mode, bla, arr, like puzzle"
31,31,1717,258,Golf With Your Friends,0.03,"hate hate, dont reccomend, game ahh, wanna kill, schizophrenia, mike, israel, game woke"
17,17,1670,267,The Binding of Isaac: Rebirth,0.02,"spam spam, farm farm, fuck abby, make crazy, buy dont, skip yes, في, semen"
1,1,1597,254,Slay the Spire 2,0.02,"massive massive, sheva, insect, flask, aeonglass, collect materials, new killer, cognition"
28,28,1521,224,Esoteric Ebb,0.05,"kill person, person kill, joel, jodie, y7, ellie, abby, esoteric ebb"
2,2,1419,166,Onimusha: Way of the Sword,0.05,"yakumo, witch time, play ryu, burrowing, garbage garbage, ryu, hayabusa, genma"
25,25,1367,184,The Last of Us™ Part II Remast,0.04,"abby, joel, omori, kasuga, unfortunately story, jodie, undertale, cliffhanger"


In [30]:
def read_cluster(c, n=10, how="typical", chars=600, seed=0):
    m = meta[meta.cluster == c]
    pick = m.sort_values("dist").head(n) if how == "typical" else m.sample(min(n, len(m)), random_state=seed)
    print(f"cluster {c}: {len(m):,} reviews from {m.appid.nunique()} games ({how})\n")
    for k, r in enumerate(pick.itertuples(), 1):
        text = r.review.strip().replace("\n", " ")
        print(f"[{k}] {r.name[:30]} | {text[:chars]}{'…' if len(text) > chars else ''}\n")
read_cluster(15, how = "random")

cluster 15: 439 reviews from 154 games (random)

[1] Summoners War | cant even get past the tutorial as the game gets stuck on a black screen

[2] Terraria | Got this game refunded not necessarily because its bad, but because the "tutorial" and early game are absolute pain to go through. I feel like if i have to watch a video or look up something about the game to be able to play it then its not a very good game, but then again I'm weird that way. Unless you enjoy overly complicated games that you have to search up to learn to simply play I'd just save your money and buy Minecraft instead.

[3] Wreckfest 2 | As of this date, this game doesnt have the basics in place, I couldnt get my steering wheel working on this. Refunding

[4] Out of Ore | It feels like the devs are a bunch of sadists making you jump through hoops for their amusement just to accomplish the most basic of tasks.  Simply connecting a water pump required googling details, and even then I struggled to figure out how to d

In [6]:
notes = {
    # cluster: (decision, evidence)
    11: ("Price & value", "Core: Planet Coaster 2 | not fun for the price and cant refund", "random: 6/10 about money or wasting money, all reviews negative"),
    18: ("Bugs & glitches", "Core: ARK: Survival Evolved | Theres too many glitches and zero support.", "random: 9/10 about glitches or bus"),
    19: ("Boring, repetitive & lacks depth", "Core: Far Far West | I purchased and played the game for around 8 hours, but my playtime was spread out intermittently. Unfortunately, I found the game very disappointing. The levels feel repetitive, there is very little enjoyment or variety, and the gameplay does not offer meaningful differences as you progress. The same scenario and gameplay elements are repeated throughout the game, which made the experience feel monotonous.", "random: 9/10 are negative surrounding the amount of time played"),
    24: ("Multiplayer & co-op design (noisy cluster)", "...", "random: 2/10 about multiplayer; core shows the topic, the cluster is mostly general complaints"),    31: ("Not a topic. Joke or hyperbolic reviews.", "Core: Rust | if you try this game you ruin your life", "random: general reviews, 1 pos, 9 neg, 3 are hyperbolic"),
    38: ("Not a topic", "Core: Borderlands 4 | I'm a big fan of previous parts, but this one feels off. UI, characters, vibe are just a miss.", "random: general negative reviews"),
    5: ("Boring/not fun, not enjoyable", "Core: Big Ambitions | This game is a lot of fun but you can max everything out very quickly, then it becomes boring. The game is way to easy to optimize. Otherwise, a pretty good game.", "9/10 about how the game is boring or unenjoyable"),
    20: ("Not a topic", "MOUSE: P.I. For Hire | This game is a good concept but executed poorly.   What it did right; Visuals look great, Voice acting is great, and you get to murder cultists.   Where it went wrong; The combat is repetitive, the guns don't have punch, the story is forgettable, the gameplay doesn't flow well, and the game gets stale fast.", "random: all a general dislike with no repeating topics")
}
notes["political"] = ("Politics & social themes (topic)",
    "grep 1.6%, 10/20 relevant, est. ~X/2%; mostly objections to 'woke'/DEI content (6), also the developer's or "
    "owner's politics (2), sexualized designs (1), poorly written politics (1); anti-Christian references in the "
    "clone grep. Covers game content and company conduct, in any direction")
notes["dead game"] = ("Merge into Multiplayer & co-op design (as 'too few players')",
    "grep 0.9%, 5/20 relevant, est. 0.2% (below 0.5%)")
notes["worse than predecessor"] = ("Worse than predecessor or a copy (topic; not patch-fixable, left out of week 7)",
    "grep 3.7%, 10/20 relevant, est. 1.9%; mostly sequels vs. earlier entries, some genre knockoffs; "
    "'rip off' also matched price complaints")

In [7]:
def grep(pattern, n=20, seed=0):
    m = meta[meta.review.str.contains(pattern, case=False, regex=True, na=False)]
    print(f"{len(m):,} of {len(meta):,} sampled negative reviews ({len(m) / len(meta):.1%}); "
          f"top clusters: {m.cluster.value_counts().head(5).to_dict()}\n")
    for k, r in enumerate(m.sample(min(n, len(m)), random_state=seed).itertuples(), 1):
        print(f"[{k}] c{r.cluster} {r.name[:25]} | {r.review.strip()[:400]!r}\n")

In [8]:
# 1. Political or social themes (either direction)
grep(r"\bwoke\b|\bdei\b|\besg\b|politic|agenda|propaganda|virtue signal|feminis|diversity|representation|censor|fan ?service|sexuali[sz]")

729 of 45,633 sampled negative reviews (1.6%); top clusters: {28: 190, 30: 68, 38: 55, 4: 47, 26: 36}

[1] c38 CODE VEIN II | 'it bad. like genuinely bad. i been playing for at least 3 hours and now notice the patten localizer language they using to in their  "PoLItICAl MeSSaGE" and honestly this is worse than that Miss kobayashi\'s dragon maid english Dub!\n\nat this point i ain\'t touching that game i bought and now i gonna ignored it.'

[2] c36 Stronghold Crusader: Defi | "Stronghold Crusader, but in Definitive Edition. A game of childhood memory. It's exactly as I remember it.\nLike, seriously, it's the same simple RTS it was 20 years ago. Units are still unbalanced as they were and gameplay improvements is laughable. Wall and moat mazes work like always. Fire is still too powerful. Horse Archers, especially paired with Mantlets, are still the fastest and cheapest o"

[3] c16 PGA TOUR 2K25 | '2k stop hiring people that graduated from a college in the past  6 years wtf are these cus

In [9]:
# 2. Dead game / too few players
grep(r"dead game|game is dead|no ?one plays|nobody plays|no players|empty (server|lobb)|ghost ?town|player ?(base|count)|queue times?")

395 of 45,633 sampled negative reviews (0.9%); top clusters: {30: 54, 24: 54, 26: 39, 32: 33, 16: 31}

[1] c26 AION 2 | "Its lacking needed mechanics and content but its okay i guess. I would say this game is about a 6 outta 10. It's super dumb you can't actually play with your friends by co-oping story missions/ quest as you progress in a f**kin MMO? Really👀. First thing you can actually do with a friend is not until level 22 which is the expedition as they call it. Another dumb idea is not having tiered loot rando"

[2] c26 Tom Clancy's Rainbow Six  | 'I began playing siege back in like 2019 and played on and off for many years i came back about once every year or so and would play for a short period of time. I have seem the rise and fall of many metas many operators and the downward spiral of this game in general. I will just say it Tom Clancys Rainbow 6 Siege is one of the most unfun unbalanced and broken games out to date that has a large pla'

[3] c26 WARDOGS | 'Wardogs used to b

C:\Users\dillo\AppData\Local\Temp\ipykernel_50600\3957146391.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = meta[meta.review.str.contains(pattern, case=False, regex=True, na=False)]


In [10]:
# 3. A worse clone / dated / worse than the predecessor
grep(r"\bclone\b|rip ?off|knock-?off|copy of|predecessor|previous (game|one|entry)|the first (game|one)|outdated|feels dated|feels old")    

C:\Users\dillo\AppData\Local\Temp\ipykernel_50600\3957146391.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = meta[meta.review.str.contains(pattern, case=False, regex=True, na=False)]


1,704 of 45,633 sampled negative reviews (3.7%); top clusters: {4: 233, 28: 209, 20: 149, 26: 146, 25: 107}

[1] c19 Wreckfest 2 | 'played for 10 minutes.. realised the first one is still better and refunded it.  maybe in a few months ill give it another go.'

[2] c11 Old World | 'Its okay if you can buy it for very cheap, but in the end its just a cheap knockoff from Civ, doing things worse.'

[3] c28 Age of Empires IV: Annive | "An absolute impudence, I don't even know where to start.\n\nWhile Age of Empires 2 suggested hours of fun within a few minutes that I was waiting to discover, I didn't find anything worth discovering in Age of Empires 4 within 40 minutes.\n\nterrible UI: nothing is placed where it should be, bad contrast between fonts and backgrounds\nweird empire's: besides the fact that most interesting empire's are "

[4] c28 The Outlast Trials | 'I was hyped when I played the first game in this series — genuinely excited, genuinely enjoying myself. So when the second game

## Draft complaint topics (2026-10-30)

From k-means (k=40) on 45,633 negative non-short reviews (at most 200 per game), each cluster read twice (10 typical,
10 random), plus keyword checks for topics spread across clusters. Counts are reviews out of 10 random that name the problem.

**Technical:** 1 Crashes (c3 10/10, c33) · 2 Won't launch / compatibility (c0 8/10, c18) · 3 Performance &
optimization (c7 10/10, c33) · 4 Bugs & glitches (c22 10/10, c18) · 5 Lost saves & progress (c35 10/10) · 6 Servers
& connectivity (c23) · 7 Accounts, launchers, DRM & privacy (c8 7/10) · 8 Controls, camera & UI (c27 10/10, c39)

**Business:** 9 Price & value (c12, c11) · 10 Monetization & pay-to-win (c16, c11) · 11 DLC practices (c29 8/10)

**Players & developers:** 12 Cheaters, toxicity & moderation (c30) · 13 Multiplayer & co-op design, including too few
players (c24, c23) · 14 Updates & developer support (c32) · 15 Generative AI (c21, noisy) · 16 Politics & social themes
(keyword est. 0.8%; game content or company conduct, any direction)

**Design:** 17 Boring, repetitive, grind or too little content (c10 10/10, c13, c37, c19, c5) · 18 Difficulty, balance
& combat, including bad enemy AI (c2 7/10, c1, c36, c26) · 19 Story, writing & characters (c25 7/10, c28) · 20 Tutorial
& onboarding (c15) · 21 Worse than predecessor or a copy (keyword est. 1.9%; not patch-fixable, left out of week 7)

**Not topics:** general rage and memes (c14, c17, c31), not in English (c9, c34; 8+ languages), genre clusters (c6, c20,
c4 partly), refunds (an outcome, D20), "dead game" (est. 0.2%, merged into 13).

**Two labelers' counts differed on 4 of 8 clusters** (c5, c18, c19, c24), because of boundaries: bugs vs. won't run
vs. other technical; boredom vs. grind vs. mentions of time; what counts as a multiplayer complaint. Fix these in the
definitions (D23) before labeling.